In [4]:
# Install required package
!pip install nba_api

import json
import time
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics, playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, playergamelogs
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Initialize DvP data
def initialize_dvp_data():
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            season="2024-25",
            season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        stats['TEAM_ABBREVIATION'] = stats['TEAM_NAME'].map(TEAM_NAME_TO_ABBREVIATION)

        dvp_data = {}
        for pos in ['C', 'F', 'G']:
            column = f'OPP_PTS_{pos.upper()}'
            if column in stats.columns:
                dvp_data[pos] = stats.groupby('TEAM_ABBREVIATION')[column].mean().to_dict()

        return dvp_data
    except Exception as e:
        print(f"Error fetching DvP stats: {e}")
        return {}

# Normalize player positions
def normalize_player_position(position):
    if not position or not isinstance(position, str):
        return []

    positions = position.split("-")
    valid_positions = {'C', 'F', 'G'}
    return [pos.strip() for pos in positions if pos.strip() in valid_positions]

# Initialize globally for consistent usage across functions
normalized_positions = []  # Collect all normalized positions globally

# Process team rosters
def process_team_roster(team_name):
    global normalized_positions  # Ensure this modifies the global list
    try:
        team_id = teams.find_team_by_abbreviation(team_name)['id']
        roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

        for _, player in roster.iterrows():
            player_name = player['PLAYER']
            player_position = player['POSITION']
            player_normalized_positions = normalize_player_position(player_position)
            normalized_positions.extend(player_normalized_positions)  # Collect positions
            print(f"Processed {player_name} with positions {player_normalized_positions}")
    except Exception as e:
        print(f"Error processing roster for team {team_name}: {e}")

# Adjust main analysis function
def analyze_teams_for_today():
    global normalized_positions  # To accumulate across games
    normalized_positions.clear()  # Reset at the start of the day's analysis

    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    if not games_today:
        print("No games scheduled for today.")
        return

    all_players_data = []

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        for team_name in [home_team, away_team]:
            process_team_roster(team_name)

    for pos in set(normalized_positions):  # Use unique positions only
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']
            for opposing_team in [home_team, away_team]:
                opposing_metrics = team_metrics.get(opposing_team, {})
                dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics.get('TEAM_ABBREVIATION', ''), 1)
                print(f"DvP adjustment for position {pos} against {opposing_team}: {dvp_factor}")

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        return playergamelog.PlayerGameLog(player_id=player_id, season="2024-25").get_data_frames()[0]
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

# Fetch injury data
def fetch_injury_data():
    try:
        url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injuries = {}
        for league in data.get('leagues', []):
            for team in league.get('teams', []):
                team_abbr = TEAM_NAME_TO_ABBREVIATION.get(team['team']['displayName'], None)
                if team_abbr:
                    for player in team.get('injuries', []):
                        injuries[player['athlete']['displayName']] = player.get('description', 'Unknown')
        return injuries
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

# Calculate stats
def calculate_stats(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv

# Adjust predictions for DvP and other factors
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data):
    try:
        positions = player_position.split("-")
        dvp_factors = [dvp_data[pos].get(opposing_metrics['TEAM_ABBREVIATION'], 1) for pos in positions if pos in dvp_data]

        dvp_factor = sum(dvp_factors) / len(dvp_factors) if dvp_factors else 1
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        return stat_mean * pace_factor * offensive_factor / defensive_factor * dvp_factor
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Main analysis for today's games
def analyze_teams_for_today():
    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    if not games_today:
        print("No games scheduled for today.")
        return

    all_players_data = []
    normalized_positions = []  # Initialize here to store all normalized positions

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        for team_name in [home_team, away_team]:
            try:
                team_id = teams.find_team_by_abbreviation(team_name)['id']
                roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                for _, player in roster.iterrows():
                    player_position = player['POSITION']
                    player_normalized_positions = normalize_player_position(player_position)
                    normalized_positions.extend(player_normalized_positions)  # Collect positions

                    # Process player as needed, e.g., fetching game logs
            except Exception as e:
                print(f"Error processing team roster for {team_name}: {e}")

    # Adjust for multi-position players in DvP adjustment
    for pos in set(normalized_positions):  # Use unique positions only
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']
            for opposing_team in [home_team, away_team]:
                opposing_metrics = team_metrics.get(opposing_team, {})
                dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics.get('TEAM_ABBREVIATION', ''), 1)
                print(f"DvP adjustment for position {pos} against {opposing_team}: {dvp_factor}")

                # Continue processing
                if home_team not in team_metrics or away_team not in team_metrics:
                        print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
                        continue

                for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
                        try:
                            # Fetch team roster here
                            team_id = teams.find_team_by_abbreviation(team_name)['id']
                            roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                            for _, player in roster.iterrows():
                                player_id = player['PLAYER_ID']
                                player_name = player['PLAYER']
                                player_position = player['POSITION']
                                usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                                injury_status = injury_data.get(player_name, 'Healthy')

                                game_logs = fetch_player_game_logs(player_id)
                                if game_logs.empty:
                                    continue

                                mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                                mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                                mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                                adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)

                                # Add new difference columns
                                ppg_diff = mean_ppg - std_dev_ppg
                                rpg_diff = mean_rpg - std_dev_rpg
                                apg_diff = mean_apg - std_dev_apg

                                all_players_data.append({
                                    'Player': player_name,
                                    'Team': team_name,
                                    'PPG': round(mean_ppg, 2),
                                    'PPG SD': round(std_dev_ppg, 2),
                                    'PPG CV': round(cv_ppg, 2),
                                    'PPG Diff': round(ppg_diff, 2),  # Added PPG Diff
                                    'Adj PPG': round(adjusted_ppg, 2),
                                    'RPG': round(mean_rpg, 2),
                                    'RPG SD': round(std_dev_rpg, 2),
                                    'RPG CV': round(cv_rpg, 2),
                                    'RPG Diff': round(rpg_diff, 2),  # Added RPG Diff
                                    'Adj RPG': round(adjusted_rpg, 2),
                                    'APG': round(mean_apg, 2),
                                    'APG SD': round(std_dev_apg, 2),
                                    'APG CV': round(cv_apg, 2),
                                    'APG Diff': round(apg_diff, 2),  # Added APG Diff
                                    'Adj APG': round(adjusted_apg, 2),
                                    'USG%': usg_pct,
                                    'Injury Status': injury_status
                                })

# Debug: Print player data added to all_players_data
                                print(f"Added data for player: {player_name}, Team: {team_name}")

                                time.sleep(1)  # To avoid rate-limiting
                        except Exception as e:
                            print(f"Error analyzing team {team_name}: {e}")
                            continue

        df = pd.DataFrame(all_players_data)
        df = df.drop_duplicates(subset=['Player', 'Team'])
        display(df)
        return df  # This is now correctly placed within the function

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.disable_dataframe_formatter()


No games scheduled for 2025-11-12.
No games scheduled for today.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
# Install required package
!pip install nba_api

import json
import time
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics, playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, playergamelogs
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Initialize DvP data
def initialize_dvp_data():
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            season="2024-25",
            season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        stats['TEAM_ABBREVIATION'] = stats['TEAM_NAME'].map(TEAM_NAME_TO_ABBREVIATION)

        dvp_data = {}
        for pos in ['C', 'F', 'G']:
            column = f'OPP_PTS_{pos.upper()}'
            if column in stats.columns:
                dvp_data[pos] = stats.groupby('TEAM_ABBREVIATION')[column].mean().to_dict()

        return dvp_data
    except Exception as e:
        print(f"Error fetching DvP stats: {e}")
        return {}

# Normalize player positions
def normalize_player_position(position):
    if not position or not isinstance(position, str):
        return []

    positions = position.split("-")
    valid_positions = {'C', 'F', 'G'}
    return [pos.strip() for pos in positions if pos.strip() in valid_positions]

# Initialize globally for consistent usage across functions
normalized_positions = []  # Collect all normalized positions globally

# Process team rosters
def process_team_roster(team_name):
    global normalized_positions  # Ensure this modifies the global list
    try:
        team_id = teams.find_team_by_abbreviation(team_name)['id']
        roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

        for _, player in roster.iterrows():
            player_name = player['PLAYER']
            player_position = player['POSITION']
            player_normalized_positions = normalize_player_position(player_position)
            normalized_positions.extend(player_normalized_positions)  # Collect positions
            print(f"Processed {player_name} with positions {player_normalized_positions}")
    except Exception as e:
        print(f"Error processing roster for team {team_name}: {e}")

# Adjust main analysis function
def analyze_teams_for_today():
    global normalized_positions  # To accumulate across games
    normalized_positions.clear()  # Reset at the start of the day's analysis

    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    if not games_today:
        print("No games scheduled for today.")
        return

    all_players_data = []

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        for team_name in [home_team, away_team]:
            process_team_roster(team_name)

    for pos in set(normalized_positions):  # Use unique positions only
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']
            for opposing_team in [home_team, away_team]:
                opposing_metrics = team_metrics.get(opposing_team, {})
                dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics.get('TEAM_ABBREVIATION', ''), 1)
                print(f"DvP adjustment for position {pos} against {opposing_team}: {dvp_factor}")

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        return playergamelog.PlayerGameLog(player_id=player_id, season="2024-25").get_data_frames()[0]
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

# Fetch injury data
def fetch_injury_data():
    try:
        url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injuries = {}
        for league in data.get('leagues', []):
            for team in league.get('teams', []):
                team_abbr = TEAM_NAME_TO_ABBREVIATION.get(team['team']['displayName'], None)
                if team_abbr:
                    for player in team.get('injuries', []):
                        injuries[player['athlete']['displayName']] = player.get('description', 'Unknown')
        return injuries
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

# Calculate stats
def calculate_stats(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv

# Adjust predictions for DvP and other factors
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data):
    try:
        positions = player_position.split("-")
        dvp_factors = [dvp_data[pos].get(opposing_metrics['TEAM_ABBREVIATION'], 1) for pos in positions if pos in dvp_data]

        dvp_factor = sum(dvp_factors) / len(dvp_factors) if dvp_factors else 1
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        return stat_mean * pace_factor * offensive_factor / defensive_factor * dvp_factor
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Main analysis for today's games
def analyze_teams_for_today():
    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    if not games_today:
        print("No games scheduled for today.")
        return

    all_players_data = []
    normalized_positions = []  # Initialize here to store all normalized positions

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        for team_name in [home_team, away_team]:
            try:
                team_id = teams.find_team_by_abbreviation(team_name)['id']
                roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                for _, player in roster.iterrows():
                    player_position = player['POSITION']
                    player_normalized_positions = normalize_player_position(player_position)
                    normalized_positions.extend(player_normalized_positions)  # Collect positions

                    # Process player as needed, e.g., fetching game logs
            except Exception as e:
                print(f"Error processing team roster for {team_name}: {e}")

    # Adjust for multi-position players in DvP adjustment
    for pos in set(normalized_positions):  # Use unique positions only
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']
            for opposing_team in [home_team, away_team]:
                opposing_metrics = team_metrics.get(opposing_team, {})
                dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics.get('TEAM_ABBREVIATION', ''), 1)
                print(f"DvP adjustment for position {pos} against {opposing_team}: {dvp_factor}")

                # Continue processing
                if home_team not in team_metrics or away_team not in team_metrics:
                        print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
                        continue

                for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
                        try:
                            # Fetch team roster here
                            team_id = teams.find_team_by_abbreviation(team_name)['id']
                            roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                            for _, player in roster.iterrows():
                                player_id = player['PLAYER_ID']
                                player_name = player['PLAYER']
                                player_position = player['POSITION']
                                usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                                injury_status = injury_data.get(player_name, 'Healthy')

                                game_logs = fetch_player_game_logs(player_id)
                                if game_logs.empty:
                                    continue

                                mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                                mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                                mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                                adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)

                                # Add new difference columns
                                low = mean_ppg - std_dev_ppg
                                high = mean_ppg + std_dev_ppg
                                ppg_diff = f"{low:.2f} - {high:.2f}"
                                low = mean_rpg - std_dev_rpg
                                high = mean_rpg + std_dev_rpg
                                rpg_diff = f"{low:.2f} - {high:.2f}"
                                low = mean_apg - std_dev_apg
                                high = mean_apg + std_dev_apg
                                apg_diff = f"{low:.2f} - {high:.2f}"

                                all_players_data.append({
                                    'Player': player_name,
                                    'Team': team_name,
                                    'PPG': round(mean_ppg, 2),
                                    'PPG SD': round(std_dev_ppg, 2),
                                    'PPG CV': round(cv_ppg, 2),
                                    'PPG Diff': ppg_diff,
                                    'Adj PPG': round(adjusted_ppg, 2),
                                    'RPG': round(mean_rpg, 2),
                                    'RPG SD': round(std_dev_rpg, 2),
                                    'RPG CV': round(cv_rpg, 2),
                                    'RPG Diff': rpg_diff,
                                    'Adj RPG': round(adjusted_rpg, 2),
                                    'APG': round(mean_apg, 2),
                                    'APG SD': round(std_dev_apg, 2),
                                    'APG CV': round(cv_apg, 2),
                                    'APG Diff': apg_diff,
                                    'Adj APG': round(adjusted_apg, 2),
                                    'USG%': usg_pct,
                                    'Injury Status': injury_status
                                })

# Debug: Print player data added to all_players_data
                                print(f"Added data for player: {player_name}, Team: {team_name}")

                                time.sleep(1)  # To avoid rate-limiting
                        except Exception as e:
                            print(f"Error analyzing team {team_name}: {e}")
                            continue

        df = pd.DataFrame(all_players_data)
        df = df.drop_duplicates(subset=['Player', 'Team'])
        df.columns = [col.center(10) for col in df.columns]
        df = df.apply(lambda row: row.map(lambda x: str(x).center(10)), axis=1)
        display(df)
        return df  # This is now correctly placed within the function

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.disable_dataframe_formatter()


No games scheduled for 2025-11-12.
No games scheduled for today.


In [ ]:
# Install required package
!pip install nba_api

import json
import time
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics, playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, playergamelogs
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Initialize DvP data
def initialize_dvp_data():
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            season="2024-25",
            season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        stats['TEAM_ABBREVIATION'] = stats['TEAM_NAME'].map(TEAM_NAME_TO_ABBREVIATION)

        dvp_data = {}
        for pos in ['C', 'F', 'G']:
            column = f'OPP_PTS_{pos.upper()}'
            if column in stats.columns:
                dvp_data[pos] = stats.groupby('TEAM_ABBREVIATION')[column].mean().to_dict()

        return dvp_data
    except Exception as e:
        print(f"Error fetching DvP stats: {e}")
        return {}

# Normalize player positions
def normalize_player_position(position):
    if not position or not isinstance(position, str):
        return []

    positions = position.split("-")
    valid_positions = {'C', 'F', 'G'}
    return [pos.strip() for pos in positions if pos.strip() in valid_positions]

# Initialize globally for consistent usage across functions
normalized_positions = []  # Collect all normalized positions globally

# Process team rosters
def process_team_roster(team_name):
    global normalized_positions  # Ensure this modifies the global list
    try:
        team_id = teams.find_team_by_abbreviation(team_name)['id']
        roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

        for _, player in roster.iterrows():
            player_name = player['PLAYER']
            player_position = player['POSITION']
            player_normalized_positions = normalize_player_position(player_position)
            normalized_positions.extend(player_normalized_positions)  # Collect positions
            print(f"Processed {player_name} with positions {player_normalized_positions}")
    except Exception as e:
        print(f"Error processing roster for team {team_name}: {e}")

# Adjust main analysis function
def analyze_teams_for_today():
    global normalized_positions  # To accumulate across games
    normalized_positions.clear()  # Reset at the start of the day's analysis

    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    if not games_today:
        print("No games scheduled for today.")
        return

    all_players_data = []

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        for team_name in [home_team, away_team]:
            process_team_roster(team_name)

    for pos in set(normalized_positions):  # Use unique positions only
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']
            for opposing_team in [home_team, away_team]:
                opposing_metrics = team_metrics.get(opposing_team, {})
                dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics.get('TEAM_ABBREVIATION', ''), 1)
                print(f"DvP adjustment for position {pos} against {opposing_team}: {dvp_factor}")

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        return playergamelog.PlayerGameLog(player_id=player_id, season="2024-25").get_data_frames()[0]
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

import requests

# ESPN API for NBA injuries
url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"

# Fetch the data
response = requests.get(url)
injury_data = response.json()

# Dictionary to store player injury statuses
injury_status_dict = {}

# Extract and store injury details
for team in injury_data.get("injuries", []):  # Loop through teams with injury reports
    team_name = team.get("team", {}).get("displayName", "Unknown Team")

    for player in team.get("players", []):  # Loop through injured players
        athlete = player.get("athlete", {})
        player_name = athlete.get("displayName", "Unknown Player")

        # Default to Healthy unless injury info is present
        injury_status = "Healthy"
        injury_type = player.get("type", {}).get("description", "")  # Example: "Out"
        injury_details = player.get("details", {})

        # Extract injury specifics if available
        if injury_type:
            injury_status = injury_type.capitalize()  # Convert "out" to "Out"

        if "type" in injury_details and "detail" in injury_details:
            injury_desc = f"{injury_details['type']} ({injury_details['detail']})"
            injury_status += f" - {injury_desc}"  # Example: "Out - Shoulder (Surgery)"

        # Store the player's injury status
        injury_status_dict[player_name] = injury_status

# Mark all missing players as Healthy (You might need a full roster source)
# This assumes you have a complete list of NBA players somewhere (e.g., API or CSV)
# Example:
# all_nba_players = ["LeBron James", "Kevin Durant", "Stephen Curry", ...]
# for player in all_nba_players:
#     if player not in injury_status_dict:
#         injury_status_dict[player] = "Healthy"

# Print all player statuses
for player, status in injury_status_dict.items():
    print(f"{player}: {status}")

# Calculate stats
def calculate_stats(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv

# Adjust predictions for DvP and other factors
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data):
    try:
        positions = player_position.split("-")
        dvp_factors = [dvp_data[pos].get(opposing_metrics['TEAM_ABBREVIATION'], 1) for pos in positions if pos in dvp_data]

        dvp_factor = sum(dvp_factors) / len(dvp_factors) if dvp_factors else 1
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        return stat_mean * pace_factor * offensive_factor / defensive_factor * dvp_factor
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Main analysis for today's games
def analyze_teams_for_today():
    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = injury_status_dict  # Store all injury statuses

    if not games_today:
        print("No games scheduled for today.")
        return

    all_players_data = []
    normalized_positions = []  # Initialize here to store all normalized positions

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        for team_name in [home_team, away_team]:
            try:
                team_id = teams.find_team_by_abbreviation(team_name)['id']
                roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                for _, player in roster.iterrows():
                    player_position = player['POSITION']
                    player_normalized_positions = normalize_player_position(player_position)
                    normalized_positions.extend(player_normalized_positions)  # Collect positions

                    # Process player as needed, e.g., fetching game logs
            except Exception as e:
                print(f"Error processing team roster for {team_name}: {e}")

    # Adjust for multi-position players in DvP adjustment
    for pos in set(normalized_positions):  # Use unique positions only
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']
            for opposing_team in [home_team, away_team]:
                opposing_metrics = team_metrics.get(opposing_team, {})
                dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics.get('TEAM_ABBREVIATION', ''), 1)
                print(f"DvP adjustment for position {pos} against {opposing_team}: {dvp_factor}")

                # Continue processing
                if home_team not in team_metrics or away_team not in team_metrics:
                        print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
                        continue

                for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
                        try:
                            # Fetch team roster here
                            team_id = teams.find_team_by_abbreviation(team_name)['id']
                            roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                            for _, player in roster.iterrows():
                                player_id = player['PLAYER_ID']
                                player_name = player['PLAYER']
                                player_position = player['POSITION']
                                usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                                injury_status = injury_data.get(player_name, 'Healthy')

                                game_logs = fetch_player_game_logs(player_id)
                                if game_logs.empty:
                                    continue

                                mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                                mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                                mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                                adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)

                                # Add new difference columns
                                low = mean_ppg - std_dev_ppg
                                high = mean_ppg + std_dev_ppg
                                ppg_diff = f"{low:.2f} - {high:.2f}"
                                low = mean_rpg - std_dev_rpg
                                high = mean_rpg + std_dev_rpg
                                rpg_diff = f"{low:.2f} - {high:.2f}"
                                low = mean_apg - std_dev_apg
                                high = mean_apg + std_dev_apg
                                apg_diff = f"{low:.2f} - {high:.2f}"

                                all_players_data.append({
                                    'Player': player_name,
                                    'Team': team_name,
                                    'PPG': round(mean_ppg, 2),
                                    'PPG SD': round(std_dev_ppg, 2),
                                    'PPG CV': round(cv_ppg, 2),
                                    'PPG Diff': ppg_diff,
                                    'Adj PPG': round(adjusted_ppg, 2),
                                    'RPG': round(mean_rpg, 2),
                                    'RPG SD': round(std_dev_rpg, 2),
                                    'RPG CV': round(cv_rpg, 2),
                                    'RPG Diff': rpg_diff,
                                    'Adj RPG': round(adjusted_rpg, 2),
                                    'APG': round(mean_apg, 2),
                                    'APG SD': round(std_dev_apg, 2),
                                    'APG CV': round(cv_apg, 2),
                                    'APG Diff': apg_diff,
                                    'Adj APG': round(adjusted_apg, 2),
                                    'USG%': usg_pct,
                                    'Injury Status': injury_status
                                })

# Debug: Print player data added to all_players_data
                                print(f"Added data for player: {player_name}, Team: {team_name}")

                                time.sleep(1)  # To avoid rate-limiting
                        except Exception as e:
                            print(f"Error analyzing team {team_name}: {e}")
                            continue

        df = pd.DataFrame(all_players_data)
        df = df.drop_duplicates(subset=['Player', 'Team'])
        df.columns = [col.center(10) for col in df.columns]
        df = df.apply(lambda row: row.map(lambda x: str(x).center(10)), axis=1)
        display(df)
        return df  # This is now correctly placed within the function

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.disable_dataframe_formatter()


Error processing team roster for DAL: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
DvP adjustment for position G against CHA: 1
Added data for player: Miles Bridges, Team: CHA
Added data for player: LaMelo Ball, Team: CHA
Added data for player: Grant Williams, Team: CHA
Added data for player: DaQuan Jeffries, Team: CHA
Added data for player: Mark Williams, Team: CHA
Added data for player: Nick Smith Jr., Team: CHA
Added data for player: Josh Green, Team: CHA
Added data for player: Cody Martin, Team: CHA
Added data for player: Josh Okogie, Team: CHA
Added data for player: Moussa Diabate, Team: CHA
Added data for player: Isaiah Wong, Team: CHA
Added data for player: Vasilije Micić, Team: CHA
Added data for player: Tre Mann, Team: CHA
Added data for player: Brandon Miller, Team: CHA
Added data for player: KJ Simpson, Team: CHA
Added data for player: Seth Curry, Team: CHA
Added data for player: Tidjane Salaun, Team: CHA
Added data for player: Taj G

,Player,Team,PPG,PPG SD,PPG CV,PPG Diff,Adj PPG,RPG,RPG SD,RPG CV,RPG Diff,Adj RPG,APG,APG SD,APG CV,APG Diff,Adj APG,USG%,Injury Status
0,Miles Bridges,CHA,18.9,6.22,0.33,12.67 - 25.12,17.37,7.34,2.47,0.34,4.88 - 9.81,6.75,3.66,1.9,0.52,1.76 - 5.55,3.36,0.254,Healthy
1,LaMelo Ball,CHA,28.16,8.84,0.31,19.32 - 37.00,25.89,5.26,2.68,0.51,2.58 - 7.94,4.83,7.29,2.78,0.38,4.51 - 10.07,6.7,0.365,Healthy
2,Grant Williams,CHA,10.38,4.43,0.43,5.95 - 14.80,9.54,5.12,2.39,0.47,2.73 - 7.52,4.71,2.31,1.54,0.66,0.78 - 3.85,2.13,0.147,Healthy
3,DaQuan Jeffries,CHA,6.42,4.68,0.73,1.74 - 11.10,5.9,3.33,2.23,0.67,1.10 - 5.56,3.06,0.83,0.94,1.12,-0.10 - 1.77,0.77,0.15,Healthy
4,Mark Williams,CHA,15.45,8.2,0.53,7.25 - 23.65,14.2,9.35,3.95,0.42,5.40 - 13.30,8.59,2.5,1.43,0.57,1.07 - 3.93,2.3,0.223,Healthy
5,Nick Smith Jr.,CHA,6.0,5.77,0.96,0.23 - 11.77,5.52,1.68,1.68,1.0,0.00 - 3.36,1.54,1.56,1.78,1.14,-0.22 - 3.34,1.43,0.198,Healthy
6,Josh Green,CHA,7.44,5.16,0.69,2.28 - 12.60,6.84,2.85,1.68,0.59,1.17 - 4.54,2.62,1.44,1.25,0.87,0.19 - 2.68,1.32,0.113,Healthy
7,Cody Martin,CHA,7.85,4.63,0.59,3.21 - 12.48,7.21,4.51,1.99,0.44,2.53 - 6.50,4.15,2.28,1.67,0.73,0.61 - 3.95,2.1,0.143,Healthy
8,Josh Okogie,CHA,6.77,5.92,0.87,0.85 - 12.68,6.22,3.07,2.86,0.93,0.20 - 5.93,2.82,0.73,0.98,1.34,-0.25 - 1.71,0.67,0.18,Healthy
9,Moussa Diabate,CHA,3.81,3.79,1.0,0.01 - 7.60,3.5,6.53,4.27,0.65,2.26 - 10.79,6.0,0.67,0.93,1.39,-0.26 - 1.59,0.61,0.095,Healthy


In [ ]:
import requests

# API URL for NBA injury data
url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"

# Send a GET request to the API
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    data = response.json()

    # Print player name and status for each injury
    for team in data.get('injuries', []):
        for injury in team.get('injuries', []):
            player_name = injury.get('athlete', {}).get('displayName', 'Unknown Player')
            status = injury.get('status', 'Unknown Status')
            print(f"Player: {player_name}, Status: {status}")
else:
    print("Failed to retrieve injury data.")


Player: Bogdan Bogdanovic, Status: Out
Player: Clint Capela, Status: Out
Player: Trae Young, Status: Day-To-Day
Player: Jalen Johnson, Status: Out
Player: Cody Zeller, Status: Out
Player: Kobe Bufkin, Status: Out
Player: Sam Hauser, Status: Out
Player: Al Horford, Status: Out
Player: Tyrese Martin, Status: Out
Player: Noah Clowney, Status: Out
Player: Cam Thomas, Status: Out
Player: Cameron Johnson, Status: Out
Player: Bojan Bogdanovic, Status: Out
Player: Maxwell Lewis, Status: Out
Player: Torrey Craig, Status: Out
Player: Zach LaVine, Status: Out
Player: Sam Merrill, Status: Out
Player: Darius Garland, Status: Out
Player: Caris LeVert, Status: Out
Player: Isaac Okoro, Status: Out
Player: Dean Wade, Status: Out
Player: Luke Travers, Status: Out
Player: Dante Exum, Status: Out
Player: Dwight Powell, Status: Out
Player: Maxi Kleber, Status: Out
Player: Dereck Lively II, Status: Out
Player: Luka Doncic, Status: Out
Player: Vlatko Cancar, Status: Out
Player: DaRon Holmes II, Status: Out
P

In [ ]:
# Install required package
!pip install nba_api

import json
import time
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics, playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, playergamelogs
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Initialize DvP data
def initialize_dvp_data():
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            season="2024-25",
            season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        stats['TEAM_ABBREVIATION'] = stats['TEAM_NAME'].map(TEAM_NAME_TO_ABBREVIATION)

        dvp_data = {}
        for pos in ['C', 'F', 'G']:
            column = f'OPP_PTS_{pos.upper()}'
            if column in stats.columns:
                dvp_data[pos] = stats.groupby('TEAM_ABBREVIATION')[column].mean().to_dict()

        return dvp_data
    except Exception as e:
        print(f"Error fetching DvP stats: {e}")
        return {}

# Normalize player positions
def normalize_player_position(position):
    if not position or not isinstance(position, str):
        return []

    positions = position.split("-")
    valid_positions = {'C', 'F', 'G'}
    return [pos.strip() for pos in positions if pos.strip() in valid_positions]

# Initialize globally for consistent usage across functions
normalized_positions = []  # Collect all normalized positions globally

# Process team rosters
def process_team_roster(team_name):
    global normalized_positions  # Ensure this modifies the global list
    try:
        team_id = teams.find_team_by_abbreviation(team_name)['id']
        roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

        for _, player in roster.iterrows():
            player_name = player['PLAYER']
            player_position = player['POSITION']
            player_normalized_positions = normalize_player_position(player_position)
            normalized_positions.extend(player_normalized_positions)  # Collect positions
            print(f"Processed {player_name} with positions {player_normalized_positions}")
    except Exception as e:
        print(f"Error processing roster for team {team_name}: {e}")

# Adjust main analysis function
def analyze_teams_for_today():
    global normalized_positions  # To accumulate across games
    normalized_positions.clear()  # Reset at the start of the day's analysis

    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    if not games_today:
        print("No games scheduled for today.")
        return

    all_players_data = []

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        for team_name in [home_team, away_team]:
            process_team_roster(team_name)

    for pos in set(normalized_positions):  # Use unique positions only
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']
            for opposing_team in [home_team, away_team]:
                opposing_metrics = team_metrics.get(opposing_team, {})
                dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics.get('TEAM_ABBREVIATION', ''), 1)
                print(f"DvP adjustment for position {pos} against {opposing_team}: {dvp_factor}")

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        return playergamelog.PlayerGameLog(player_id=player_id, season="2024-25").get_data_frames()[0]
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

def fetch_injury_data():
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injury_status_dict = {}  # Store injuries in {player_name: status} format

        for team in data.get('injuries', []):
            for injury in team.get('injuries', []):
                player_name = injury.get('athlete', {}).get('displayName', 'Unknown Player')
                status = injury.get('status', 'Healthy')  # Default to 'Healthy'
                injury_status_dict[player_name] = status

        return injury_status_dict
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

# Calculate stats
def calculate_stats(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv

# Adjust predictions for DvP and other factors
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data):
    try:
        positions = player_position.split("-")
        dvp_factors = [dvp_data[pos].get(opposing_metrics['TEAM_ABBREVIATION'], 1) for pos in positions if pos in dvp_data]

        dvp_factor = sum(dvp_factors) / len(dvp_factors) if dvp_factors else 1
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        return stat_mean * pace_factor * offensive_factor / defensive_factor * dvp_factor
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Main analysis for today's games
def analyze_teams_for_today():
    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_status_dict = fetch_injury_data()  # Assuming this function fetches injury data
    injury_data = injury_status_dict  # Store all injury statuses

    if not games_today:
        print("No games scheduled for today.")
        return

    all_players_data = []
    normalized_positions = []  # Initialize here to store all normalized positions

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        for team_name in [home_team, away_team]:
            try:
                team_id = teams.find_team_by_abbreviation(team_name)['id']
                roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                for _, player in roster.iterrows():
                    player_position = player['POSITION']
                    player_normalized_positions = normalize_player_position(player_position)
                    normalized_positions.extend(player_normalized_positions)  # Collect positions

                    # Process player as needed, e.g., fetching game logs
            except Exception as e:
                print(f"Error processing team roster for {team_name}: {e}")

    # Adjust for multi-position players in DvP adjustment
    for pos in set(normalized_positions):  # Use unique positions only
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']
            for opposing_team in [home_team, away_team]:
                opposing_metrics = team_metrics.get(opposing_team, {})
                dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics.get('TEAM_ABBREVIATION', ''), 1)
                print(f"DvP adjustment for position {pos} against {opposing_team}: {dvp_factor}")

                # Continue processing
                if home_team not in team_metrics or away_team not in team_metrics:
                        print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
                        continue

                for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
                        try:
                            # Fetch team roster here
                            team_id = teams.find_team_by_abbreviation(team_name)['id']
                            roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                            # Fetch injury data once to use throughout the script
                            injury_data = fetch_injury_data()

                            for _, player in roster.iterrows():
                                player_id = player['PLAYER_ID']
                                player_name = player['PLAYER']
                                player_position = player['POSITION']
                                usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                                injury_status = injury_data.get(player_name, 'Healthy')

                                game_logs = fetch_player_game_logs(player_id)
                                if game_logs.empty:
                                    continue

                                mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                                mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                                mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                                adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)

                                # Add new difference columns
                                low = mean_ppg - std_dev_ppg
                                high = mean_ppg + std_dev_ppg
                                ppg_diff = f"{low:.2f} - {high:.2f}"
                                low = mean_rpg - std_dev_rpg
                                high = mean_rpg + std_dev_rpg
                                rpg_diff = f"{low:.2f} - {high:.2f}"
                                low = mean_apg - std_dev_apg
                                high = mean_apg + std_dev_apg
                                apg_diff = f"{low:.2f} - {high:.2f}"

                                all_players_data.append({
                                    'Player': player_name,
                                    'Position': player_normalized_positions,
                                    'Team': team_name,
                                    'PPG': round(mean_ppg, 2),
                                    'PPG SD': round(std_dev_ppg, 2),
                                    'PPG CV': round(cv_ppg, 2),
                                    'PPG Diff': ppg_diff,
                                    'Adj PPG': round(adjusted_ppg, 2),
                                    'RPG': round(mean_rpg, 2),
                                    'RPG SD': round(std_dev_rpg, 2),
                                    'RPG CV': round(cv_rpg, 2),
                                    'RPG Diff': rpg_diff,
                                    'Adj RPG': round(adjusted_rpg, 2),
                                    'APG': round(mean_apg, 2),
                                    'APG SD': round(std_dev_apg, 2),
                                    'APG CV': round(cv_apg, 2),
                                    'APG Diff': apg_diff,
                                    'Adj APG': round(adjusted_apg, 2),
                                    'USG%': usg_pct,
                                    'Injury Status': injury_status
                                })

# Debug: Print player data added to all_players_data
                                print(f"Added data for player: {player_name}, Team: {team_name}")

                                time.sleep(1)  # To avoid rate-limiting
                        except Exception as e:
                            print(f"Error analyzing team {team_name}: {e}")
                            continue

        df = pd.DataFrame(all_players_data)
        df = df.drop_duplicates(subset=['Player', 'Team'])
        df.columns = [col.center(10) for col in df.columns]
        df = df.apply(lambda row: row.map(lambda x: str(x).center(10)), axis=1)
        display(df)
        return df  # This is now correctly placed within the function

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.disable_dataframe_formatter()


DvP adjustment for position C against WAS: 1
Added data for player: Bilal Coulibaly, Team: WAS
Added data for player: Tristan Vukcevic, Team: WAS
Added data for player: Johnny Davis, Team: WAS
Added data for player: Jared Butler, Team: WAS
Added data for player: Patrick Baldwin Jr., Team: WAS
Added data for player: Bub Carrington, Team: WAS
Added data for player: Justin Champagnie, Team: WAS
Added data for player: Jordan Poole, Team: WAS
Added data for player: Malcolm Brogdon, Team: WAS
Added data for player: Anthony Gill, Team: WAS
Added data for player: Jonas Valančiūnas, Team: WAS
Added data for player: Kyshawn George, Team: WAS
Added data for player: Alex Sarr, Team: WAS
Added data for player: Richaun Holmes, Team: WAS
Added data for player: Corey Kispert, Team: WAS
Added data for player: Kyle Kuzma, Team: WAS
Added data for player: Marvin Bagley III, Team: WAS
Added data for player: Jalen Hood-Schifino, Team: LAL
Added data for player: Jarred Vanderbilt, Team: LAL
Added data for p

,Player,Position,Team,PPG,PPG SD,PPG CV,PPG Diff,Adj PPG,RPG,RPG SD,RPG CV,RPG Diff,Adj RPG,APG,APG SD,APG CV,APG Diff,Adj APG,USG%,Injury Status
0,Bilal Coulibaly,['G'],WAS,12.31,6.7,0.54,5.61 - 19.01,11.53,5.0,2.28,0.46,2.72 - 7.28,4.68,3.36,1.95,0.58,1.41 - 5.31,3.14,0.177,Healthy
1,Tristan Vukcevic,['G'],WAS,1.88,3.48,1.86,-1.61 - 5.36,1.76,0.62,0.74,1.19,-0.12 - 1.37,0.59,0.38,0.74,1.98,-0.37 - 1.12,0.35,0.244,Healthy
2,Johnny Davis,['G'],WAS,2.44,2.82,1.16,-0.38 - 5.25,2.28,1.12,1.58,1.41,-0.46 - 2.71,1.05,0.34,0.55,1.59,-0.20 - 0.89,0.32,0.156,Healthy
3,Jared Butler,['G'],WAS,6.84,5.91,0.86,0.93 - 12.75,6.4,1.32,1.74,1.32,-0.42 - 3.06,1.24,2.55,2.06,0.81,0.49 - 4.61,2.39,0.253,Healthy
4,Patrick Baldwin Jr.,['G'],WAS,2.1,2.07,0.99,0.02 - 4.17,1.96,0.95,1.24,1.31,-0.29 - 2.20,0.89,0.1,0.3,3.16,-0.21 - 0.40,0.09,0.155,Healthy
5,Bub Carrington,['G'],WAS,8.78,4.74,0.54,4.05 - 13.52,8.23,4.09,2.39,0.59,1.69 - 6.48,3.83,3.78,2.06,0.55,1.72 - 5.85,3.54,0.144,Healthy
6,Justin Champagnie,['G'],WAS,8.11,7.53,0.93,0.57 - 15.64,7.59,4.89,3.44,0.7,1.46 - 8.33,4.58,1.04,1.2,1.16,-0.17 - 2.24,0.97,0.155,Healthy
7,Jordan Poole,['G'],WAS,20.74,8.93,0.43,11.81 - 29.68,19.43,3.0,1.97,0.66,1.03 - 4.97,2.81,4.9,2.34,0.48,2.56 - 7.23,4.59,0.282,Healthy
8,Malcolm Brogdon,['G'],WAS,13.83,7.44,0.54,6.40 - 21.27,12.96,4.06,2.94,0.72,1.12 - 7.00,3.8,3.94,2.26,0.57,1.68 - 6.21,3.69,0.243,Out
9,Anthony Gill,['G'],WAS,1.94,2.21,1.14,-0.27 - 4.15,1.82,0.85,1.3,1.53,-0.45 - 2.15,0.79,0.21,0.48,2.28,-0.27 - 0.70,0.2,0.128,Healthy


In [3]:
# Install required package
!pip install nba_api

import json
import time
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics, playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, playergamelogs
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Initialize DvP data
def initialize_dvp_data():
    try:
        stats = leaguedashteamstats.LeagueDashTeamStats(
            measure_type_detailed_defense='Opponent',
            per_mode_detailed='PerGame',
            season="2024-25",
            season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        stats['TEAM_ABBREVIATION'] = stats['TEAM_NAME'].map(TEAM_NAME_TO_ABBREVIATION)

        dvp_data = {}
        for pos in ['C', 'F', 'G']:
            column = f'OPP_PTS_{pos.upper()}'
            if column in stats.columns:
                dvp_data[pos] = stats.groupby('TEAM_ABBREVIATION')[column].mean().to_dict()

        return dvp_data
    except Exception as e:
        print(f"Error fetching DvP stats: {e}")
        return {}

# Normalize player positions
def normalize_player_position(position):
    if not position or not isinstance(position, str):
        return []

    positions = position.split("-")
    valid_positions = {'C', 'F', 'G'}
    return [pos.strip() for pos in positions if pos.strip() in {"C", "F", "G"}]
    print(f"Player: {player_name}, Original Position: {player_position}, Normalized: {player_normalized_positions}")

# Initialize globally for consistent usage across functions
normalized_positions = []  # Collect all normalized positions globally

# Process team rosters
def process_team_roster(team_name):
    global normalized_positions  # Ensure this modifies the global list
    try:
        team_id = teams.find_team_by_abbreviation(team_name)['id']
        roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

        for _, player in roster.iterrows():
            player_name = player['PLAYER']
            player_position = player['POSITION']
            player_normalized_positions = normalize_player_position(player_position)
            normalized_positions.extend(player_normalized_positions)  # Collect positions
            print(f"Processed {player_name} with positions {player_normalized_positions}")
    except Exception as e:
        print(f"Error processing roster for team {team_name}: {e}")

# Adjust main analysis function
def analyze_teams_for_today():
    global normalized_positions  # To accumulate across games
    normalized_positions.clear()  # Reset at the start of the day's analysis

    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    if not games_today:
        print("No games scheduled for today.")
        return

    all_players_data = []

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        for team_name in [home_team, away_team]:
            process_team_roster(team_name)

    for pos in set(normalized_positions):  # Use unique positions only
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']
            for opposing_team in [home_team, away_team]:
                opposing_metrics = team_metrics.get(opposing_team, {})
                dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics.get('TEAM_ABBREVIATION', ''), 1)
                print(f"DvP adjustment for position {pos} against {opposing_team}: {dvp_factor}")

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        return playergamelog.PlayerGameLog(player_id=player_id, season="2024-25").get_data_frames()[0]
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

def fetch_injury_data():
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injury_status_dict = {}  # Store injuries in {player_name: status} format

        for team in data.get('injuries', []):
            for injury in team.get('injuries', []):
                player_name = injury.get('athlete', {}).get('displayName', 'Unknown Player')
                status = injury.get('status', 'Healthy')  # Default to 'Healthy'
                injury_status_dict[player_name] = status

        return injury_status_dict
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

# Calculate stats
def calculate_stats(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv

# Adjust predictions for DvP and other factors
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data):
    try:
        positions = player_position.split("-")
        dvp_factors = [dvp_data[pos].get(opposing_metrics['TEAM_ABBREVIATION'], 1) for pos in positions if pos in dvp_data]

        dvp_factor = sum(dvp_factors) / len(dvp_factors) if dvp_factors else 1
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        return stat_mean * pace_factor * offensive_factor / defensive_factor * dvp_factor
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Main analysis for today's games
def analyze_teams_for_today():
    dvp_data = initialize_dvp_data()
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_status_dict = fetch_injury_data()  # Assuming this function fetches injury data
    injury_data = injury_status_dict  # Store all injury statuses

    if not games_today:
        print("No games scheduled for today.")
        return

    all_players_data = []
    normalized_positions = []  # Initialize here to store all normalized positions

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        for team_name in [home_team, away_team]:
            try:
                team_id = teams.find_team_by_abbreviation(team_name)['id']
                roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                for _, player in roster.iterrows():
                    player_position = player['POSITION']
                    player_normalized_positions = normalize_player_position(player_position)
                    normalized_positions.extend(player_normalized_positions)  # Collect positions

                    # Process player as needed, e.g., fetching game logs
            except Exception as e:
                print(f"Error processing team roster for {team_name}: {e}")

    # Adjust for multi-position players in DvP adjustment
    for pos in set(normalized_positions):  # Use unique positions only
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']
            for opposing_team in [home_team, away_team]:
                opposing_metrics = team_metrics.get(opposing_team, {})
                dvp_factor = dvp_data.get(pos, {}).get(opposing_metrics.get('TEAM_ABBREVIATION', ''), 1)
                print(f"DvP adjustment for position {pos} against {opposing_team}: {dvp_factor}")

                # Continue processing
                if home_team not in team_metrics or away_team not in team_metrics:
                        print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
                        continue

                for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
                        try:
                            # Fetch team roster here
                            team_id = teams.find_team_by_abbreviation(team_name)['id']
                            roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                            # Fetch injury data once to use throughout the script
                            injury_data = fetch_injury_data()

                            for _, player in roster.iterrows():
                                player_id = player['PLAYER_ID']
                                player_name = player['PLAYER']
                                player_position = player['POSITION']
                                usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                                injury_status = injury_data.get(player_name, 'Healthy')

                                game_logs = fetch_player_game_logs(player_id)
                                if game_logs.empty:
                                    continue

                                mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                                mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                                mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                                adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)
                                adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position, dvp_data)

                                # Add new difference columns
                                low = mean_ppg - std_dev_ppg
                                high = mean_ppg + std_dev_ppg
                                ppg_diff = f"{low:.2f} - {high:.2f}"
                                low = mean_rpg - std_dev_rpg
                                high = mean_rpg + std_dev_rpg
                                rpg_diff = f"{low:.2f} - {high:.2f}"
                                low = mean_apg - std_dev_apg
                                high = mean_apg + std_dev_apg
                                apg_diff = f"{low:.2f} - {high:.2f}"

                                all_players_data.append({
                                    'Player': player_name,
                                    'Position': player_normalized_positions,
                                    'Team': team_name,
                                    'PPG': round(mean_ppg, 2),
                                    'PPG SD': round(std_dev_ppg, 2),
                                    'PPG CV': round(cv_ppg, 2),
                                    'PPG Diff': ppg_diff,
                                    'Adj PPG': round(adjusted_ppg, 2),
                                    'RPG': round(mean_rpg, 2),
                                    'RPG SD': round(std_dev_rpg, 2),
                                    'RPG CV': round(cv_rpg, 2),
                                    'RPG Diff': rpg_diff,
                                    'Adj RPG': round(adjusted_rpg, 2),
                                    'APG': round(mean_apg, 2),
                                    'APG SD': round(std_dev_apg, 2),
                                    'APG CV': round(cv_apg, 2),
                                    'APG Diff': apg_diff,
                                    'Adj APG': round(adjusted_apg, 2),
                                    'USG%': usg_pct,
                                    'Injury Status': injury_status
                                })

# Debug: Print player data added to all_players_data
                                print(f"Added data for player: {player_name}, Team: {team_name}")

                                time.sleep(1)  # To avoid rate-limiting
                        except Exception as e:
                            print(f"Error analyzing team {team_name}: {e}")
                            continue

        df = pd.DataFrame(all_players_data)
        df = df.drop_duplicates(subset=['Player', 'Team'])
        df.columns = [col.center(10) for col in df.columns]
        df = df.apply(lambda row: row.map(lambda x: str(x).center(10)), axis=1)
        display(df)
        return df  # This is now correctly placed within the function

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.disable_dataframe_formatter()


No games scheduled for 2025-11-12.
No games scheduled for today.


In [2]:
# Install required package
!pip install nba_api

import json
import time
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.endpoints import teamestimatedmetrics, playerestimatedmetrics
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster
from nba_api.stats.static import teams
from nba_api.stats.endpoints import playergamelog, playergamelogs
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display

# Map of team names to abbreviations
TEAM_NAME_TO_ABBREVIATION = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "LA Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS"
}

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Fetch team metrics
def fetch_team_metrics():
    try:
        stats = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        ).get_data_frames()[0]

        metrics = stats[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        team_metrics = {TEAM_NAME_TO_ABBREVIATION.get(k, k): v for k, v in metrics_dict.items()}
        return team_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Process team rosters
def process_team_roster(team_name):
    global normalized_positions  # Ensure this modifies the global list
    try:
        team_id = teams.find_team_by_abbreviation(team_name)['id']
        roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

        for _, player in roster.iterrows():
            player_name = player['PLAYER']
            player_position = player['POSITION']
            player_normalized_positions = normalize_player_position(player_position)
            normalized_positions.extend(player_normalized_positions)  # Collect positions
            print(f"Processed {player_name} with positions {player_normalized_positions}")
    except Exception as e:
        print(f"Error processing roster for team {team_name}: {e}")

# Adjust main analysis function
def analyze_teams_for_today():
    global normalized_positions  # To accumulate across games
    normalized_positions.clear()  # Reset at the start of the day's analysis
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data()

    if not games_today:
        print("No games scheduled for today.")
        return

    all_players_data = []

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        for team_name in [home_team, away_team]:
            process_team_roster(team_name)

    for pos in set(normalized_positions):  # Use unique positions only
        for game in games_today:
            home_team = game['home_team']
            away_team = game['away_team']
            for opposing_team in [home_team, away_team]:
                opposing_metrics = team_metrics.get(opposing_team, {})

# Fetch player game logs
def fetch_player_game_logs(player_id):
    try:
        return playergamelog.PlayerGameLog(player_id=player_id, season="2024-25").get_data_frames()[0]
    except Exception as e:
        print(f"Error fetching game logs for player {player_id}: {e}")
        return pd.DataFrame()

def fetch_injury_data():
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injury_status_dict = {}  # Store injuries in {player_name: status} format

        for team in data.get('injuries', []):
            for injury in team.get('injuries', []):
                player_name = injury.get('athlete', {}).get('displayName', 'Unknown Player')
                status = injury.get('status', 'Healthy')  # Default to 'Healthy'
                injury_status_dict[player_name] = status

        return injury_status_dict
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

# Calculate stats
def calculate_stats(stat_series):
    mean = stat_series.mean()
    std_dev = stat_series.std()
    cv = std_dev / mean if mean != 0 else 0
    return mean, std_dev, cv

# Adjust predictions for ORTG, DRTG, and PACE
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position):
    try:
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        return stat_mean * pace_factor * offensive_factor / defensive_factor
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Main analysis for today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_status_dict = fetch_injury_data()  # Assuming this function fetches injury data
    injury_data = injury_status_dict  # Store all injury statuses

    if not games_today:
        print("No games scheduled for today.")
        return

    all_players_data = []
    normalized_positions = []  # Initialize here to store all normalized positions

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        for team_name in [home_team, away_team]:
            try:
                team_id = teams.find_team_by_abbreviation(team_name)['id']
                roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                for _, player in roster.iterrows():
                    player_position = player['POSITION']
                    player_positions = player_position(player_position)
                    player_positions.extend(player_positions)  # Collect positions

                    # Process player as needed, e.g., fetching game logs
            except Exception as e:
                print(f"Error processing team roster for {team_name}: {e}")

                # Continue processing
                if home_team not in team_metrics or away_team not in team_metrics:
                        print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
                        continue

                for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
                        try:
                            # Fetch team roster here
                            team_id = teams.find_team_by_abbreviation(team_name)['id']
                            roster = commonteamroster.CommonTeamRoster(team_id=team_id).get_data_frames()[0]

                            # Fetch injury data once to use throughout the script
                            injury_data = fetch_injury_data()

                            for _, player in roster.iterrows():
                                player_id = player['PLAYER_ID']
                                player_name = player['PLAYER']
                                player_position = player['POSITION']
                                usg_pct = player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A')
                                injury_status = injury_data.get(player_name, 'Healthy')

                                game_logs = fetch_player_game_logs(player_id)
                                if game_logs.empty:
                                    continue

                                mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                                mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                                mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                                adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position)
                                adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position)
                                adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating, player_position)

                                # Add new difference columns
                                low = mean_ppg - std_dev_ppg
                                high = mean_ppg + std_dev_ppg
                                ppg_diff = f"{low:.2f} - {high:.2f}"
                                low = mean_rpg - std_dev_rpg
                                high = mean_rpg + std_dev_rpg
                                rpg_diff = f"{low:.2f} - {high:.2f}"
                                low = mean_apg - std_dev_apg
                                high = mean_apg + std_dev_apg
                                apg_diff = f"{low:.2f} - {high:.2f}"

                                all_players_data.append({
                                    'Player': player_name,
                                    'Position': player_position,
                                    'Team': team_name,
                                    'PPG': round(mean_ppg, 2),
                                    'PPG SD': round(std_dev_ppg, 2),
                                    'PPG CV': round(cv_ppg, 2),
                                    'PPG Diff': ppg_diff,
                                    'Adj PPG': round(adjusted_ppg, 2),
                                    'RPG': round(mean_rpg, 2),
                                    'RPG SD': round(std_dev_rpg, 2),
                                    'RPG CV': round(cv_rpg, 2),
                                    'RPG Diff': rpg_diff,
                                    'Adj RPG': round(adjusted_rpg, 2),
                                    'APG': round(mean_apg, 2),
                                    'APG SD': round(std_dev_apg, 2),
                                    'APG CV': round(cv_apg, 2),
                                    'APG Diff': apg_diff,
                                    'Adj APG': round(adjusted_apg, 2),
                                    'USG%': usg_pct,
                                    'Injury Status': injury_status
                                })

# Debug: Print player data added to all_players_data
                                print(f"Added data for player: {player_name}, Team: {team_name}")

                                time.sleep(1)  # To avoid rate-limiting
                        except Exception as e:
                            print(f"Error analyzing team {team_name}: {e}")
                            continue

        df = pd.DataFrame(all_players_data)
        df = df.drop_duplicates(subset=['Player', 'Team'])
        df.columns = [col.center(10) for col in df.columns]
        df = df.apply(lambda row: row.map(lambda x: str(x).center(10)), axis=1)
        display(df)
        return df  # This is now correctly placed within the function

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.disable_dataframe_formatter()


No games scheduled for 2025-11-12.
No games scheduled for today.


In [1]:
# Install required package
!pip install nba_api

import json
from nba_api.stats.endpoints import teamestimatedmetrics
from nba_api.stats.endpoints import TeamVsPlayer
from nba_api.stats.endpoints import PlayerGameLogs
from nba_api.stats.endpoints import playergamelog
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.static import teams
from nba_api.stats.endpoints import commonteamroster
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display
import time  # Ensure time is imported

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Set timezone to EST
        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Function to fetch DvP stats
def fetch_dvp_stats(team_id, season_type="Regular Season"):
    try:
        stats = TeamVsPlayer(team_id=team_id, season_type=season_type)
        return stats.get_data_frames()[0]  # return the first dataframe
    except Exception as e:
        print(f"Error fetching DvP stats for team {team_id}: {e}")
        return None

# Function to process player data
def process_players(team):
    team_players = team['players']
    player_stats = []
    for player in team_players:
        player_data = {
            'Player': player['full_name'],
            'Position': player.get('position', 'Unknown')  # Handle missing 'POS' field
        }
        player_stats.append(player_data)
    return player_stats

# Fetch team metrics
def fetch_team_metrics():
    try:
        team_metrics = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        )
        data = team_metrics.get_data_frames()[0]
        metrics = data[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        # Calculate league averages
        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        # Map abbreviations to full names
        team_name_mapping = {
            "ATL": "Atlanta Hawks", "BOS": "Boston Celtics", "BKN": "Brooklyn Nets", "CHA": "Charlotte Hornets",
            "CHI": "Chicago Bulls", "CLE": "Cleveland Cavaliers", "DAL": "Dallas Mavericks", "DEN": "Denver Nuggets",
            "DET": "Detroit Pistons", "GSW": "Golden State Warriors", "HOU": "Houston Rockets", "IND": "Indiana Pacers",
            "LAC": "LA Clippers", "LAL": "Los Angeles Lakers", "MEM": "Memphis Grizzlies", "MIA": "Miami Heat",
            "MIL": "Milwaukee Bucks", "MIN": "Minnesota Timberwolves", "NOP": "New Orleans Pelicans", "NYK": "New York Knicks",
            "OKC": "Oklahoma City Thunder", "ORL": "Orlando Magic", "PHI": "Philadelphia 76ers", "PHX": "Phoenix Suns",
            "POR": "Portland Trail Blazers", "SAC": "Sacramento Kings", "SAS": "San Antonio Spurs", "TOR": "Toronto Raptors",
            "UTA": "Utah Jazz", "WAS": "Washington Wizards"
        }

        # Adjusted metrics dictionary
        adjusted_metrics = {}
        for abbr, full_name in team_name_mapping.items():
            if full_name in metrics_dict:
                adjusted_metrics[abbr] = metrics_dict[full_name]
            else:
                print(f"Warning: Metrics not found for {abbr} ({full_name}).")

        return adjusted_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Fetch player estimated metrics for USG%
def fetch_player_usg():
    try:
        stats = playerestimatedmetrics.PlayerEstimatedMetrics(
            season="2024-25", season_type="Regular Season"
        ).get_data_frames()[0]

        usg_data = stats[['PLAYER_ID', 'PLAYER_NAME', 'E_USG_PCT']]
        return usg_data.set_index('PLAYER_ID').to_dict(orient='index')
    except Exception as e:
        print(f"Error fetching player usage metrics: {e}")
        return {}

# Fetch injury data
def fetch_injury_data():
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        injury_status_dict = {}  # Store injuries in {player_name: status} format

        for team in data.get('injuries', []):
            for injury in team.get('injuries', []):
                player_name = injury.get('athlete', {}).get('displayName', 'Unknown Player')
                status = injury.get('status', 'Healthy')  # Default to 'Healthy'
                injury_status_dict[player_name] = status

        return injury_status_dict
    except Exception as e:
        print(f"Error fetching injury data: {e}")
        return {}

# Define the adjust_prediction function
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating):
    try:
        # Calculate pace adjustment factor
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1

        # Offensive adjustment factor
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1

        # Defensive adjustment factor (using opponent's defensive rating)
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        # Apply adjustments to the stat mean
        adjusted_stat = stat_mean * pace_factor * offensive_factor / defensive_factor
        return adjusted_stat
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Analyze teams and players for today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()
    player_usg_data = fetch_player_usg()
    injury_data = fetch_injury_data() # Assuming this function fetches injury
    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    all_players_data = []

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        if home_team not in team_metrics or away_team not in team_metrics:
            print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
            continue

        for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
            try:
                team_roster = commonteamroster.CommonTeamRoster(
                    team_id=teams.find_team_by_abbreviation(team_name)['id']
                ).get_data_frames()[0]

                for _, player in team_roster.iterrows():
                    player_id = player['PLAYER_ID']
                    player_name = player['PLAYER']
                    print(f"Fetching data for {player_name} ({team_name})")

                    game_logs = fetch_player_game_logs(player_id)
                    if game_logs.empty:
                        continue

                    mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                    mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                    mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                    adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)

                    all_players_data.append({
                        'Player': player_name,
                        'Position': player['POSITION'],
                        'Team': team_name,
                        'PPG': round(mean_ppg,2),
                        'Adj PPG': round(adjusted_ppg, 2),
                        'PPG SD': round(std_dev_ppg, 2),
                        'PPG CV': round(cv_ppg, 2),
                        'RPG': round(mean_rpg, 2),
                        'Adj RPG': round(adjusted_rpg, 2),
                        'RPG SD': round(std_dev_rpg, 2),
                        'RPG CV': round(cv_rpg, 2),
                        'APG': round(mean_apg, 2),
                        'Adj APG': round(adjusted_apg, 2),
                        'APG SD': round(std_dev_apg, 2),
                        'APG CV': round(cv_apg, 2),
                        'USG%': player_usg_data.get(player_id, {}).get('E_USG_PCT', 'N/A'),
                        'Inj Status': injury_data.get(player_name, 'Healthy')  # Use the fetched injury status
                    })
                    # Add a sleep delay to respect API request limits
                    time.sleep(1)  # Sleep for 1 second between requests
            except Exception as e:
                print(f"Error analyzing team {team_name} or player {player_name}: {e}")
                continue

    # Display the players data
    if all_players_data:
        df = pd.DataFrame(all_players_data)
        display(df)
        return df
    else:
        return pd.DataFrame()

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.enable_dataframe_formatter()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 5.6 MB/s eta 0:00:00
No games scheduled for 2025-11-12.
Error fetching player usage metrics: name 'playerestimatedmetrics' is not defined
No games scheduled for today.
